# Model Training
Define the target and sensor features for your dataset, split the data without leakage, train a baseline estimator, and persist the selected model.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_PATH = PROJECT_ROOT / "models" / "irrigation_model.joblib"

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data: {PROCESSED_DATA_DIR}")
print(f"Model path: {MODEL_PATH}")

In [ ]:
#error showing sckit-learn is not installed, please install it using pip install scikit-learn
import sklearn
print(sklearn.__version__)

1.9.1


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [2]:
#checking the environment my notebook is running in 
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: c:\Users\Admin\AppData\Local\Programs\Python\Python314\python.exe
Python version: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]


In [ ]:
#found out that my jupyter notebook is currently using my global Python Installation and not my smart irrigation project virtual environme
# cd path\to\smart-irrigation
#activate virtua; environment


In [4]:
#error above is due to PROJECT_ROOT was referenced on line 1, but Python has no record of what PROJECT_ROOT points to 
#i need to define PROJECT_ROOT using pyhton's linrary before attenpting to use it. 

In [11]:
from pathlib import Path
import pandas as pd

# 1. Retrieve and sort files by modification time (newest first) instead of alphabetically
processed_files = sorted(
    PROCESSED_DATA_DIR.glob("*.csv"),
    key=lambda f: f.stat().st_mtime,
    reverse=True
)

if not processed_files:
    raise FileNotFoundError(
        "Run notebook 02 to create a processed CSV first."
    )

# 2. Load the latest generated dataset safely
latest_file = processed_files[0]
print(f"Loading latest processed data from: {latest_file.name}")

data = pd.read_csv(latest_file)

Loading latest processed data from: irrigation_prediction_processed.csv


In [7]:
from pathlib import Path
import pandas as pd

# Define the absolute path to your processed data directory
PROCESSED_DATA_DIR = Path(r"G:\AIOT\smart-irrigation\data\processed")

# Retrieve and sort CSV files by modification time (newest first)
processed_files = sorted(
    PROCESSED_DATA_DIR.glob("*.csv"),
    key=lambda f: f.stat().st_mtime,
    reverse=True
)

if not processed_files:
    raise FileNotFoundError(
        f"No processed CSV files found in {PROCESSED_DATA_DIR}. Run notebook 02 first."
    )

# Load the latest generated dataset
latest_file = processed_files[0]
print(f"Loading latest processed data from: {latest_file.resolve()}")

data = pd.read_csv(latest_file)

Loading latest processed data from: G:\AIOT\smart-irrigation\data\processed\irrigation_prediction_processed.csv


In [9]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_PATH = PROJECT_ROOT / "models" / "irrigation_model.joblib"

# Set these to the real target and feature columns after inspecting your dataset.
TARGET_COLUMN = 'Irrigation_Need'
processed_files = sorted(PROCESSED_DATA_DIR.glob('*.csv'))
if not processed_files:
    raise FileNotFoundError('Run notebook 02 to create a processed CSV first.')
data = pd.read_csv(processed_files[0])
if TARGET_COLUMN not in data.columns:
    raise ValueError(f'Set TARGET_COLUMN to a column in the dataset: {list(data.columns)}')
X = data.drop(columns=[TARGET_COLUMN]).select_dtypes(include='number')
y = data[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = RandomForestRegressor(random_state=42) if y.dtype.kind in 'if' else RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)
print(f'Saved model to {MODEL_PATH}; held-out rows: {len(X_test)}')

Saved model to g:\AIOT\smart-irrigation\models\irrigation_model.joblib; held-out rows: 2000


In [10]:
print(pd.read_csv(processed_files[0]).columns.tolist())


['Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare', 'Mulching_Used', 'Previous_Irrigation_mm', 'Region', 'Irrigation_Need']
